In [11]:
import pandas as pd
import re
import string
# import spacy
from chatwords import chat_words

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.stem import WordNetLemmatizer

In [12]:
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()
stop_words = stopwords.words('english')
# nlp = spacy.load("en_core_web_sm")

In [13]:
def preprocess_text(text):

    # 1 Lowercase
    text = text.lower()

    # 2 Remove HTML tags
    text = re.sub(r"<.*?>","",text)

    # 3 Remove URLs
    text = re.sub(r"http\S+|www\S+","",text)

    # 4 Remove emojis
    text = re.sub(r"[^\w\s]", "", text)

    # 5 Tokenization
    tokens = word_tokenize(text)

    # 6 Chat word treatment
    new_tokens = []
    for word in tokens:
        if word in chat_words:
            new_tokens.extend(chat_words[word].split())
        else:
            new_tokens.append(word)

    tokens = new_tokens

    # 7 Remove punctuation
    tokens = [w.translate(str.maketrans('', '', string.punctuation)) for w in tokens]

    # 8 Remove stopwords
    tokens = [word for word in tokens if word not in stop_words]

    # 9 Spelling normalization (reduce repeated characters)
    tokens = [re.sub(r"(.)\1+", r"\1\1", word) for word in tokens]

    # 10 Stemming
    tokens = [stemmer.stem(word) for word in tokens]

    # 11 Lemmatization
    tokens = [lemmatizer.lemmatize(word) for word in tokens]

    # Join tokens
    return " ".join(tokens)

In [14]:
def preprocess_spacy(text):

    # lowercase
    text = text.lower()

    # remove html
    text = re.sub(r"<.*?>", "", text)

    # remove urls
    text = re.sub(r"http\S+|www\S+", "", text)

    # chat word replacement
    for word in chat_words:
        text = text.replace(word, chat_words[word])

    # remove emojis
    text = re.sub(r"[^\w\s]", "", text)

    # spelling normalization (soooo → soo)
    text = re.sub(r"(.)\1+", r"\1\1", text)

    # spaCy processing
    doc = nlp(text)

    tokens = []

    for token in doc:

        if token.is_stop:
            continue

        if token.is_punct:
            continue

        lemma = token.lemma_

        if lemma.strip() != "":
            tokens.append(lemma)

    return " ".join(tokens)

In [15]:
df = pd.read_csv("movie_dataset_preprocessing_practice.csv")

In [16]:
df

,name,description,genre
0,Final Kingdom,I luv this film sooo much lol!!! 😂😂,Horror
1,Dark City,terrible acting omg!!! 😂😂,Comedy
2,Galaxy Mission,best film everrrrr!!! check https://filmreview...,Drama
3,Silent Hill,OMG this movie is soooo good!!! 😍😍 <br> visit ...,Adventure
4,Night Hunter,terrible acting omg!!! 😂😂,Action
...,...,...,...
9995,Ocean Fear,OMG this movie is soooo good!!! 😍😍 <br> visit ...,Horror
9996,Night Hunter,best film everrrrr!!! check https://filmreview...,Comedy
9997,Night Hunter,idk why ppl hate this movie lol it's awesome!!!,Adventure
9998,Future War,<p>This movie was amazng!!! must watch!!!</p>,Comedy


In [17]:
df["clean_description"] = df["description"].apply(preprocess_text)
# df["clean_description_spacy"] = df["clean_description"].apply(preprocess_spacy)

In [18]:
df

,name,description,genre,clean_description
0,Final Kingdom,I luv this film sooo much lol!!! 😂😂,Horror,luv film soo much lol
1,Dark City,terrible acting omg!!! 😂😂,Comedy,terribl act omg
2,Galaxy Mission,best film everrrrr!!! check https://filmreview...,Drama,best film everr check
3,Silent Hill,OMG this movie is soooo good!!! 😍😍 <br> visit ...,Adventure,omg movi soo good visit
4,Night Hunter,terrible acting omg!!! 😂😂,Action,terribl act omg
...,...,...,...,...
9995,Ocean Fear,OMG this movie is soooo good!!! 😍😍 <br> visit ...,Horror,omg movi soo good visit
9996,Night Hunter,best film everrrrr!!! check https://filmreview...,Comedy,best film everr check
9997,Night Hunter,idk why ppl hate this movie lol it's awesome!!!,Adventure,idk ppl hate movi lol awesom
9998,Future War,<p>This movie was amazng!!! must watch!!!</p>,Comedy,movi amazng must watch
